# Advanced LLM API Patterns: OpenAI & Anthropic

Covers interview-critical topics beyond basic call/response:

1. **Advanced Parameters & Configuration** — sampling, logprobs, token counting
2. **Streaming** — chunk-by-chunk output handling
3. **Tool Use / Function Calling** ← most important for interviews
4. **Multi-modal** — images via URL and base64
5. **Agentic Loops** — ReAct pattern
6. **RAG** — retrieval-augmented generation pipeline
7. **Memory & Session Management** — history strategies
8. **Structured Output & LLM-as-Judge** — schema-constrained responses

## 0. Setup & Imports

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os, json, base64, time
from typing import Any

for _root in [Path.cwd(), *Path.cwd().parents]:
    _env = _root / ".env"
    if _env.is_file():
        load_dotenv(_env)
        break

from openai import OpenAI
import anthropic

oai = OpenAI()
ant = anthropic.Anthropic()

# Use cheaper/faster models for learning
OAI_MODEL = "gpt-4o-mini"
ANT_MODEL = "claude-sonnet-4-6"

print("Clients initialized:", oai.__class__.__name__, ant.__class__.__name__)

Clients initialized: OpenAI Anthropic


---
## 1. Advanced Parameters & Configuration

Beyond `model` + `messages`, these sampling knobs control quality and behavior.

In [2]:
# ── OpenAI: full parameter breakdown ──────────────────────────────────────
response = oai.chat.completions.create(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Name 5 European capitals, numbered."}],

    # Sampling controls (use EITHER temperature OR top_p, not both)
    temperature=0.3,        # 0 = deterministic, 2 = maximally random
    top_p=0.9,              # nucleus sampling: only tokens in top 90% probability mass

    max_tokens=150,         # hard cap on output tokens
    stop=["6."],            # generation stops when this string is produced

    # Repetition controls (-2 to +2)
    presence_penalty=0.5,   # penalizes tokens that have appeared at all (encourages new topics)
    frequency_penalty=0.3,  # penalizes tokens proportional to how often they've appeared

    n=1,                    # number of completions to generate (costs n x tokens)
    seed=42,                # best-effort determinism across identical calls

    # Log probabilities — useful for uncertainty estimation
    logprobs=True,
    top_logprobs=3,         # return top 3 token alternatives at each position
)

print(response.choices[0].message.content)
print("\n--- logprob sample (first 2 tokens) ---")
for lp in response.choices[0].logprobs.content[:2]:
    import math
    print(f"  token='{lp.token}'  prob={math.exp(lp.logprob):.3f}")
    for alt in lp.top_logprobs:
        print(f"    alt: '{alt.token}'  prob={math.exp(alt.logprob):.3f}")

print("\nFinish reason:", response.choices[0].finish_reason)
print("Usage:", response.usage)

Sure! Here are five European capitals:

1. Paris (France)
2. Berlin (Germany)
3. Madrid (Spain)
4. Rome (Italy)
5. Amsterdam (Netherlands)

--- logprob sample (first 2 tokens) ---
  token='Sure'  prob=0.722
    alt: 'Sure'  prob=0.722
    alt: '1'  prob=0.265
    alt: 'Here'  prob=0.008
  token='!'  prob=1.000
    alt: '!'  prob=1.000
    alt: ','  prob=0.000
    alt: '!

'  prob=0.000

Finish reason: stop
Usage: CompletionUsage(completion_tokens=39, prompt_tokens=15, total_tokens=54, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


In [3]:
# ── Anthropic: parameter differences vs OpenAI ───────────────────────────
response = ant.messages.create(
    model=ANT_MODEL,
    max_tokens=150,         # REQUIRED in Anthropic (no default)

    # System prompt is a top-level field in Anthropic (not a messages entry)
    system="You are a concise geography assistant.",

    messages=[{"role": "user", "content": "Name 5 European capitals, numbered."}],

    # Sampling: Sonnet 4.x allows temperature OR top_p, not both (400 if both sent).
    temperature=0.3,        # 0-1 range (not 0-2 like OpenAI); omit if using top_p instead
    top_k=40,               # Anthropic-only: limit vocab to top K tokens at each step
    stop_sequences=["6."],  # list of strings (Anthropic uses 'stop_sequences' not 'stop')
)

print(response.content[0].text)
print("\nStop reason:", response.stop_reason)  # 'end_turn' | 'max_tokens' | 'stop_sequence'
print("Usage:", response.usage)               # input_tokens, output_tokens
print("Model:", response.model)

1. Paris
2. Berlin
3. Madrid
4. Rome
5. Vienna

Stop reason: end_turn
Usage: Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=25, output_tokens=23, server_tool_use=None, service_tier='standard')
Model: claude-sonnet-4-6


In [4]:
# ── Token counting ────────────────────────────────────────────────────────
# Critical for staying within context limits and estimating cost

import tiktoken

# OpenAI: tiktoken library (offline, no API call needed)
enc = tiktoken.encoding_for_model("gpt-4o-mini")
text = "How many tokens does this sentence contain?"
tokens = enc.encode(text)
print(f"OpenAI tiktoken count: {len(tokens)} tokens")
print(f"  Token IDs: {tokens[:5]}...")
print(f"  Decoded back: {enc.decode(tokens)}")

# Count a messages list (accounts for per-message overhead ~4 tokens)
def count_oai_messages(messages: list, model: str = "gpt-4o-mini") -> int:
    enc = tiktoken.encoding_for_model(model)
    total = 3  # base overhead
    for m in messages:
        total += 4  # per-message overhead
        if isinstance(m.get("content"), str):
            total += len(enc.encode(m["content"]))
    return total

msgs = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2 + 2?"},
]
print(f"\nMessage token count: {count_oai_messages(msgs)}")

# Anthropic: native API-based token counting (requires API call)
count = ant.messages.count_tokens(
    model=ANT_MODEL,
    system="You are a helpful assistant.",
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
)
print(f"Anthropic API token count: {count.input_tokens} input tokens")

OpenAI tiktoken count: 8 tokens
  Token IDs: [5299, 1991, 20290, 2226, 495]...
  Decoded back: How many tokens does this sentence contain?

Message token count: 25
Anthropic API token count: 23 input tokens


---
## 2. Streaming Responses

Streaming yields tokens as they're generated — essential for responsive UIs and agentic pipelines.

In [5]:
# ── OpenAI Streaming ──────────────────────────────────────────────────────
print("=== stream=True (manual chunk handling) ===")
with oai.chat.completions.stream(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5, one per line."}],
    max_tokens=80,
) as stream:
    for event in stream:
        # OpenAI Python SDK v2: `.stream()` yields typed events, not raw chunks.
        # Text deltas use type "content.delta" (see ContentDeltaEvent).
        if event.type == "content.delta":
            print(event.delta, end="", flush=True)

print("\n")

# ── Accumulate to final message ───────────────────────────────────────────
print("=== Accumulate to final message ===")
with oai.chat.completions.stream(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Say hello in 3 words."}],
) as stream:
    final = stream.get_final_completion()
print("Final:", final.choices[0].message.content)
print("Usage:", final.usage)

=== stream=True (manual chunk handling) ===
1  
2  
3  
4  
5  

=== Accumulate to final message ===
Final: Hello there, friend!
Usage: None


In [6]:
# ── Anthropic Streaming ───────────────────────────────────────────────────
print("=== text_stream iterator ===")
with ant.messages.stream(
    model=ANT_MODEL,
    max_tokens=80,
    messages=[{"role": "user", "content": "Count from 1 to 5, one per line."}],
) as stream:
    for text_chunk in stream.text_stream:  # yields only text deltas
        print(text_chunk, end="", flush=True)

print("\n")

# ── Raw event stream (gives access to all event types) ────────────────────
print("=== Raw event stream ===")
with ant.messages.stream(
    model=ANT_MODEL,
    max_tokens=80,
    messages=[{"role": "user", "content": "Say hi."}],
) as stream:
    # Common event types: message_start, content_block_start, content_block_delta,
    # content_block_stop, message_delta, message_stop
    for event in stream:
        print(f"  [{event.type}]", end="")
        if hasattr(event, 'delta') and hasattr(event.delta, 'text'):
            print(f" '{event.delta.text}'", end="")
        print()
    
    # After the context exits, get final message with usage stats
    final_msg = stream.get_final_message()
    print(f"\nInput tokens: {final_msg.usage.input_tokens}")
    print(f"Output tokens: {final_msg.usage.output_tokens}")

=== text_stream iterator ===
1
2
3
4
5

=== Raw event stream ===
  [message_start]
  [content_block_start]
  [content_block_delta] 'Hi'
  [text]
  [content_block_delta] ' there! 👋 How are you doing? Is there something I can help you with today?'
  [text]
  [content_block_stop]
  [message_delta]
  [message_stop]

Input tokens: 10
Output tokens: 25


---
## 3. Tool Use / Function Calling  ← Most Important for Interviews

Tool use lets the model decide when and which external function to call. The pattern:
1. Send message + tool definitions to LLM
2. LLM returns a **tool_call** (not a text response)
3. You execute the function and return the result
4. LLM receives result and produces a final text response

**Key differences between providers:**
| | OpenAI | Anthropic |
|---|---|---|  
| Schema key | `parameters` | `input_schema` |
| Tool result role | `role: "tool"` | `role: "user"` with `type: "tool_result"` |
| Finish signal | `finish_reason == "tool_calls"` | `stop_reason == "tool_use"` |
| Force tool | `tool_choice: "required"` | `tool_choice: {"type": "any"}` |
| Specific tool | `tool_choice: {"type":"function", ...}` | `tool_choice: {"type": "tool", "name": ...}` |

In [7]:
# ── Tool definitions and mock implementations ─────────────────────────────

# OpenAI tool schema format
oai_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city. Call this when the user asks about weather.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'Paris'"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"],
                             "description": "Temperature unit. Defaults to celsius."},
                },
                "required": ["city"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "Math expression string, e.g. '2 + 3 * 4'"},
                },
                "required": ["expression"],
                "additionalProperties": False,
            },
        },
    },
]

# Anthropic tool schema format — nearly identical but uses 'input_schema'
ant_tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "input_schema": {          # <-- 'input_schema' instead of 'parameters'
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["city"],
        },
    },
    {
        "name": "calculate",
        "description": "Evaluate a math expression.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"},
            },
            "required": ["expression"],
        },
    },
]

# Mock implementations
def get_weather(city: str, unit: str = "celsius") -> dict:
    db = {"london": 14, "paris": 19, "new york": 23, "tokyo": 26, "sydney": 17}
    temp = db.get(city.lower(), 20)
    if unit == "fahrenheit":
        temp = round(temp * 9 / 5 + 32, 1)
    return {"city": city, "temperature": temp, "unit": unit, "condition": "partly cloudy"}

def calculate(expression: str) -> dict:
    # NEVER use eval on untrusted user input in production!
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}

def execute_tool(name: str, args: dict) -> str:
    """Dispatch tool call to implementation, return JSON string."""
    if name == "get_weather":
        return json.dumps(get_weather(**args))
    elif name == "calculate":
        return json.dumps(calculate(**args))
    return json.dumps({"error": f"Unknown tool: {name}"})

print("Tools defined:", [t["function"]["name"] for t in oai_tools])
print("Test get_weather:", get_weather("Paris"))
print("Test calculate:", calculate("15 * 23 + 7"))

Tools defined: ['get_weather', 'calculate']
Test get_weather: {'city': 'Paris', 'temperature': 19, 'unit': 'celsius', 'condition': 'partly cloudy'}
Test calculate: {'expression': '15 * 23 + 7', 'result': 352}


In [8]:
# ── OpenAI: single-turn tool call inspection ─────────────────────────────
# Shows the raw structure of the model's tool call response

response = oai.chat.completions.create(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "What's the weather in Paris?"}],
    tools=oai_tools,
    tool_choice="auto",   # auto | none | required | {"type":"function","function":{"name":"..."}}
)

msg = response.choices[0].message
print("Finish reason:", response.choices[0].finish_reason)  # "tool_calls" when tool needed
print("Content:", msg.content)           # None when tool is called
print("Tool calls:", msg.tool_calls)

if msg.tool_calls:
    tc = msg.tool_calls[0]
    print(f"\nTool call id:   {tc.id}")
    print(f"Function name:  {tc.function.name}")
    print(f"Arguments:      {tc.function.arguments}")  # always a JSON string
    args = json.loads(tc.function.arguments)
    print(f"Parsed args:    {args}")
    result = execute_tool(tc.function.name, args)
    print(f"Tool result:    {result}")

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_R8xH1IIo6oeJqz6EAFQpMTNt', function=Function(arguments='{"city":"Paris"}', name='get_weather'), type='function')]

Tool call id:   call_R8xH1IIo6oeJqz6EAFQpMTNt
Function name:  get_weather
Arguments:      {"city":"Paris"}
Parsed args:    {'city': 'Paris'}
Tool result:    {"city": "Paris", "temperature": 19, "unit": "celsius", "condition": "partly cloudy"}


In [9]:
# ── OpenAI: full multi-turn tool loop ─────────────────────────────────────
# The pattern every agentic system uses

def oai_tool_loop(user_query: str, verbose: bool = True, max_iters: int = 6) -> str:
    messages = [{"role": "user", "content": user_query}]

    for i in range(max_iters):
        response = oai.chat.completions.create(
            model=OAI_MODEL,
            messages=messages,
            tools=oai_tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        # Append assistant message object directly (preserves tool_calls field)
        messages.append(msg)

        if response.choices[0].finish_reason == "tool_calls":
            # Execute all tool calls from this turn
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                result = execute_tool(tc.function.name, args)
                if verbose:
                    print(f"  [tool] {tc.function.name}({args}) → {result}")
                # Tool result message: role='tool', must include tool_call_id
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,   # CRITICAL: must match the call's id
                    "content": result,        # must be a string
                })
        else:
            # finish_reason == 'stop' — model is done
            return msg.content

    return "Max iterations reached"

print("Query: What's the weather in Paris and London, and what's 15 * 23?")
print("─" * 60)
answer = oai_tool_loop("What's the weather in Paris and London, and what's 15 * 23?")
print("─" * 60)
print("Answer:", answer)

Query: What's the weather in Paris and London, and what's 15 * 23?
────────────────────────────────────────────────────────────
  [tool] get_weather({'city': 'Paris'}) → {"city": "Paris", "temperature": 19, "unit": "celsius", "condition": "partly cloudy"}
  [tool] get_weather({'city': 'London'}) → {"city": "London", "temperature": 14, "unit": "celsius", "condition": "partly cloudy"}
  [tool] calculate({'expression': '15 * 23'}) → {"expression": "15 * 23", "result": 345}
────────────────────────────────────────────────────────────
Answer: The weather is as follows:

- **Paris**: 19°C, partly cloudy
- **London**: 14°C, partly cloudy

Additionally, the result of \( 15 \times 23 \) is **345**.


In [10]:
# ── OpenAI: parallel tool calls & forced tool use ─────────────────────────

# Parallel: model automatically requests multiple tools in one response
response = oai.chat.completions.create(
    model=OAI_MODEL,
    messages=[{"role": "user",
               "content": "Compare weather in Tokyo and New York, and also compute 100 / 4"}],
    tools=oai_tools,
)
msg = response.choices[0].message
print(f"Number of parallel tool calls: {len(msg.tool_calls)}")
for tc in msg.tool_calls:
    print(f"  [{tc.id[:12]}...] {tc.function.name}({tc.function.arguments})")

print()

# Forced tool use: model MUST call at least one tool (useful for structured extraction)
response = oai.chat.completions.create(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],  # would NOT normally use a tool
    tools=oai_tools,
    tool_choice="required",   # force at least one tool call
)
print("Forced (required) — tool called:",
      response.choices[0].message.tool_calls[0].function.name)

# Force a specific tool (ignores what the model thinks is best)
response = oai.chat.completions.create(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "What is the weather? Also do some math."}],
    tools=oai_tools,
    tool_choice={"type": "function", "function": {"name": "calculate"}},  # specific tool
)
print("Forced specific tool:",
      response.choices[0].message.tool_calls[0].function.name)  # always 'calculate'

Number of parallel tool calls: 3
  [call_vwXupKY...] get_weather({"city": "Tokyo"})
  [call_QIZvBIr...] get_weather({"city": "New York"})
  [call_kfl8IBl...] calculate({"expression": "100 / 4"})

Forced (required) — tool called: get_weather
Forced specific tool: calculate


In [11]:
# ── Anthropic: full tool loop ─────────────────────────────────────────────
# Key differences vs OpenAI:
#  - assistant message content is a LIST of blocks (TextBlock | ToolUseBlock)
#  - tool results go in a USER message with 'tool_result' content blocks
#  - ALL tool results for a turn go in a single user message

def ant_tool_loop(user_query: str, verbose: bool = True, max_iters: int = 6) -> str:
    messages = [{"role": "user", "content": user_query}]

    for _ in range(max_iters):
        response = ant.messages.create(
            model=ANT_MODEL,
            max_tokens=1024,
            tools=ant_tools,
            tool_choice={"type": "auto"},  # "auto" | "any" | {"type":"tool","name":"..."}
            messages=messages,
        )

        # Append assistant turn (content is a list of TextBlock/ToolUseBlock)
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "tool_use":
            # Collect ALL tool results into a single user-turn content list
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    if verbose:
                        print(f"  [tool] {block.name}({block.input}) → {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,   # CRITICAL: must match the ToolUseBlock id
                        "content": result,          # string or list of content blocks
                    })
            # All results go in one user message
            messages.append({"role": "user", "content": tool_results})
        else:
            # stop_reason == 'end_turn'
            return next((b.text for b in response.content if hasattr(b, "text")), "")

    return "Max iterations reached"

print("Query: What's the weather in Paris and what's 42 * 7?")
print("─" * 60)
answer = ant_tool_loop("What's the weather in Paris and what's 42 * 7?")
print("─" * 60)
print("Answer:", answer)

Query: What's the weather in Paris and what's 42 * 7?
────────────────────────────────────────────────────────────
  [tool] get_weather({'city': 'Paris'}) → {"city": "Paris", "temperature": 19, "unit": "celsius", "condition": "partly cloudy"}
  [tool] calculate({'expression': '42 * 7'}) → {"expression": "42 * 7", "result": 294}
────────────────────────────────────────────────────────────
Answer: Here are your answers:

1. 🌤️ **Weather in Paris:** It's currently **19°C** and **partly cloudy**.
2. 🔢 **42 × 7 = 294**

Let me know if you need anything else!


In [12]:
# ── Anthropic: forced tool use ────────────────────────────────────────────

# Force at least one tool (type='any')
response = ant.messages.create(
    model=ANT_MODEL,
    max_tokens=256,
    tools=ant_tools,
    tool_choice={"type": "any"},    # force model to use at least one tool
    messages=[{"role": "user", "content": "Hello there!"}],
)
print("Forced 'any' — tool called:", response.content[0].name)
print("Input:", response.content[0].input)

print()

# Force a specific tool (type='tool' + name)
response = ant.messages.create(
    model=ANT_MODEL,
    max_tokens=256,
    tools=ant_tools,
    tool_choice={"type": "tool", "name": "calculate"},  # force specific tool
    messages=[{"role": "user", "content": "What's the weather like today?"}],
)
print("Forced specific tool:", response.content[0].name)  # always 'calculate'
print("Input:", response.content[0].input)   # model fills in what it thinks makes sense

print()

# Disable tools entirely
response = ant.messages.create(
    model=ANT_MODEL,
    max_tokens=256,
    tools=ant_tools,
    tool_choice={"type": "none"},   # model will NOT use any tools
    messages=[{"role": "user", "content": "What's the weather in Paris?"}],
)
print("tool_choice='none' — response:", response.content[0].text)

Forced 'any' — tool called: calculate
Input: {'expression': '1+1'}

Forced specific tool: calculate
Input: {'expression': '1+1'}

tool_choice='none' — response: Let me check the current weather in Paris for you!


---
## 4. Multi-modal Inputs

Both providers accept images inline with text. Two delivery methods:
- **URL** — provider fetches the image (simpler, but image must be public)
- **Base64** — encode locally and embed in request (works for private images)

In [13]:
import base64
import urllib.request
from pathlib import Path
from PIL import Image
import io

# Relative path: ml_evals/assets/sample_multimodal.png (folder is created if missing).
# Works when Jupyter cwd is `prototype/` or `prototype/ml_evals/`.
_cwd = Path.cwd()
if (_cwd / "llm_api_advanced.ipynb").is_file():
    _assets = _cwd / "assets"
elif (_cwd / "ml_evals" / "llm_api_advanced.ipynb").is_file():
    _assets = _cwd / "ml_evals" / "assets"
else:
    _assets = _cwd / "assets"
_assets.mkdir(parents=True, exist_ok=True)
IMG_PATH = _assets / "sample_multimodal.png"

# Set False to use your own PNG at IMG_PATH (skips download).
DOWNLOAD_SAMPLE = True

if DOWNLOAD_SAMPLE:
    # Wikimedia: (1) send a real User-Agent (default Python is often 403), and
    # (2) use a direct /commons/... file URL — /thumb/.../NNNpx-... can return 400 (thumbnail policy).
    IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png"
    _req = urllib.request.Request(
        IMG_URL,
        headers={"User-Agent": "prototype-llm-notebook/1.0 (educational use; Python urllib)"},
    )
    with urllib.request.urlopen(_req) as resp, open(IMG_PATH, "wb") as out:
        out.write(resp.read())

# Load and encode to base64
with open(IMG_PATH, "rb") as f:
    IMG_B64 = base64.standard_b64encode(f.read()).decode("utf-8")

img = Image.open(IMG_PATH)
print(f"Image: {img.size} px, mode={img.mode}, path={IMG_PATH.resolve()}, base64 length={len(IMG_B64)} chars")

Image: (800, 600) px, mode=RGBA, path=/Users/larryjin/Documents/Programs/prototype/ml_evals/assets/sample_multimodal.png, base64 length=299424 chars


In [14]:
# ── OpenAI Vision ─────────────────────────────────────────────────────────
# Content is a list of content parts: text + image_url

# Method 1: URL (image fetched by OpenAI servers)
response = oai.chat.completions.create(
    model=OAI_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image briefly."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": IMG_URL,
                        "detail": "low",   # 'low' (~85 tokens) | 'high' (tiles) | 'auto'
                    },
                },
            ],
        }
    ],
)
print("URL vision:", response.choices[0].message.content)

# Method 2: Base64 (data URI)
response = oai.chat.completions.create(
    model=OAI_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What are the dominant colors in this image?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{IMG_B64}",  # data URI format
                    },
                },
            ],
        }
    ],
)
print("Base64 vision:", response.choices[0].message.content)

URL vision: The image features three colorful dice: a blue die, a red die, and a green die, each with white dots representing numbers. They appear to be rolling or bouncing, creating a sense of movement against a colorful, blurred background.
Base64 vision: The dominant colors in the image are blue, red, green, and yellow, along with white for the dots on the dice.


In [15]:
# ── Anthropic Vision ──────────────────────────────────────────────────────
# Content block uses 'image' type with a 'source' dict

# Method 1: Base64 source
response = ant.messages.create(
    model=ANT_MODEL,
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",  # image/jpeg | image/gif | image/webp
                        "data": IMG_B64,
                    },
                },
                {"type": "text", "text": "Describe this image in one sentence."},
            ],
        }
    ],
)
print("Anthropic base64:", response.content[0].text)

# Method 2: URL source — Anthropic downloads this URL from *their* servers (not your laptop).
# Many hosts block those fetches or disallow crawlers in robots.txt → 400 errors (e.g.
# "Unable to download" or "disallowed by robots.txt"). placehold.co disallows bots.
# This demo uses httpbin's PNG (robots.txt only blocks /deny). Prefer base64 for reliability.
VISION_URL_DEMO = "https://httpbin.org/image/png"

response = ant.messages.create(
    model=ANT_MODEL,
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {"type": "url", "url": VISION_URL_DEMO},
                },
                {"type": "text", "text": "What is shown in this image?"},
            ],
        }
    ],
)
print("Anthropic URL:", response.content[0].text)

# Multiple images in one prompt
response = ant.messages.create(
    model=ANT_MODEL,
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Image 1:"},
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": IMG_B64}},
                {"type": "text", "text": "Image 2:"},
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": IMG_B64}},
                {"type": "text", "text": "Are these two images identical?"},
            ],
        }
    ],
)
print("Multi-image:", response.content[0].text)

Anthropic base64: Four colorful translucent dice — **blue, green, red, and yellow** — are captured mid-air as if being tossed, against a white background.
Anthropic URL: # 🐷 Pig Face

The image shows a **cartoon/illustrated pig face** with:

- **Round pink face**
- **Small round ears** on top
- **Black dot eyes**
- **Pink rosy cheeks** (blush marks)
- **Snout/nose** in the center
- **Curved smile**

It appears to be a simple, cute **emoji-style pig illustration**, similar to the 🐷 pig face emoji!
Multi-image: ## Comparison of the Two Images

After careful examination, the two images are **nearly identical**, but there is a subtle difference:

- **Image 1**: The **red die** appears to be positioned **slightly higher** (more elevated/closer to the viewer) compared to Image 2.
- **Image 2**: The **red die** appears to be positioned **slightly lower**.

The blue, green, and yellow dice appear to be in the **same positions** in both images. The overall composition, colors, lighting, and sty

---
## 5. Agentic Loops — ReAct Pattern

**ReAct** = **Re**asoning + **Act**ing. The model interleaves:
- **Thought**: "I need to find the weather first, then do the math"
- **Action**: call `get_weather(city='Paris')`
- **Observation**: `{"temperature": 19, ...}`
- **Thought**: "Now I need to calculate..."

This is the backbone of frameworks like LangChain, LlamaIndex, and AutoGen.

In [16]:
# ── ReAct Agent (Anthropic) ───────────────────────────────────────────────
# Extended tools with a search mock for richer agent behavior

from tenacity import retry, stop_after_attempt, wait_exponential

agent_tools_ant = ant_tools + [
    {
        "name": "search_web",
        "description": "Search the web for current information on a topic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
            },
            "required": ["query"],
        },
    },
]

def search_web(query: str) -> dict:
    return {"results": [f"Mocked search result for: {query}",
                        "LLMs have context windows measured in tokens."]}

def execute_agent_tool(name: str, args: dict) -> str:
    if name == "get_weather":
        return json.dumps(get_weather(**args))
    elif name == "calculate":
        return json.dumps(calculate(**args))
    elif name == "search_web":
        return json.dumps(search_web(**args))
    return json.dumps({"error": f"Unknown tool: {name}"})

REACT_SYSTEM = """You are a helpful assistant with access to tools.
Think step-by-step before acting. Use tools when you need external information.
After getting tool results, synthesize them into a clear final answer."""

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=4))
def react_agent(query: str, max_iters: int = 8, verbose: bool = True) -> str:
    messages = [{"role": "user", "content": query}]
    iteration = 0

    while iteration < max_iters:
        iteration += 1
        if verbose:
            print(f"\n--- Iteration {iteration} ---")

        response = ant.messages.create(
            model=ANT_MODEL,
            max_tokens=1024,
            system=REACT_SYSTEM,
            tools=agent_tools_ant,
            messages=messages,
        )

        messages.append({"role": "assistant", "content": response.content})

        # Print what the model thought/said
        if verbose:
            for block in response.content:
                if hasattr(block, "text"):
                    print(f"[Thought] {block.text[:300]}")
                elif block.type == "tool_use":
                    print(f"[Action]  {block.name}({json.dumps(block.input)})")

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_agent_tool(block.name, block.input)
                    if verbose:
                        print(f"[Observe] {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            return next((b.text for b in response.content if hasattr(b, "text")), "")

    return "Max iterations reached"

query = "What's the combined temperature of Paris and Tokyo in Celsius, and what is that sum divided by 2?"
print(f"Query: {query}")
print("=" * 60)
final = react_agent(query)
print("\n" + "=" * 60)
print("Final answer:", final)

Query: What's the combined temperature of Paris and Tokyo in Celsius, and what is that sum divided by 2?

--- Iteration 1 ---
[Thought] I'll fetch the current temperatures for both Paris and Tokyo simultaneously!
[Action]  get_weather({"city": "Paris", "unit": "celsius"})
[Action]  get_weather({"city": "Tokyo", "unit": "celsius"})
[Observe] {"city": "Paris", "temperature": 19, "unit": "celsius", "condition": "partly cloudy"}
[Observe] {"city": "Tokyo", "temperature": 26, "unit": "celsius", "condition": "partly cloudy"}

--- Iteration 2 ---
[Thought] Got the temperatures! Now let me calculate the sum and the average at the same time.
[Action]  calculate({"expression": "19 + 26"})
[Action]  calculate({"expression": "(19 + 26) / 2"})
[Observe] {"expression": "19 + 26", "result": 45}
[Observe] {"expression": "(19 + 26) / 2", "result": 22.5}

--- Iteration 3 ---
[Thought] Here's a full breakdown:

| City | Temperature |
|-------|------------|
| 🇫🇷 Paris | 19°C (partly cloudy) |
| 🇯🇵 Tokyo |

---
## 6. RAG — Retrieval-Augmented Generation

RAG pipeline:
1. **Index**: embed a document corpus into vectors
2. **Retrieve**: embed the query, find nearest-neighbor chunks by cosine similarity
3. **Augment**: inject retrieved chunks into the LLM prompt as context
4. **Generate**: LLM answers grounded in retrieved facts

This avoids hallucination and keeps the model's knowledge current without fine-tuning.

In [17]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Sample knowledge base (would be chunked documents in production)
DOCS = [
    {"id": 0, "text": "Transformer models use self-attention to process all tokens in parallel, enabling parallelism during training."},
    {"id": 1, "text": "RAG (Retrieval-Augmented Generation) combines a retriever with an LLM generator to ground responses in external documents."},
    {"id": 2, "text": "Fine-tuning updates all model weights on task-specific data; LoRA adapts only small rank-decomposition matrices."},
    {"id": 3, "text": "LLM evaluation metrics include BLEU, ROUGE for n-gram overlap, BERTScore for semantic similarity, and LLM-as-judge for quality."},
    {"id": 4, "text": "Embeddings map text to dense vectors where semantic similarity corresponds to proximity in vector space."},
    {"id": 5, "text": "Chain-of-thought prompting improves model reasoning by asking it to show intermediate steps before the final answer."},
    {"id": 6, "text": "Context window limits drive chunking strategies: fixed-size, sentence-based, or semantic chunking."},
    {"id": 7, "text": "RLHF trains a reward model on human preferences, then uses PPO to align the LLM with those preferences."},
    {"id": 8, "text": "Temperature controls output randomness: 0 is greedy/deterministic, higher values increase diversity."},
    {"id": 9, "text": "Prompt caching stores reusable prefix KV cache on the provider side, cutting latency and cost for repeated long contexts."},
]

# Build embedding index
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_texts = [d["text"] for d in DOCS]
doc_embs = embedder.encode(doc_texts, normalize_embeddings=True, show_progress_bar=False)
print(f"Index built: {len(DOCS)} docs, embedding dim={doc_embs.shape[1]}")

def retrieve(query: str, k: int = 3) -> list[dict]:
    """Return top-k docs by cosine similarity (normalized vectors → dot product = cosine)."""
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores = (doc_embs @ q_emb.T).squeeze()
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [{"text": DOCS[i]["text"], "score": float(scores[i])} for i in top_k_idx]

# Test retrieval
test_query = "How do I evaluate the quality of LLM outputs?"
chunks = retrieve(test_query)
print(f"\nQuery: {test_query}")
for c in chunks:
    print(f"  [{c['score']:.3f}] {c['text'][:80]}...")

/Users/larryjin/Documents/Programs/anaconda3/envs/prototype/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19616.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index built: 10 docs, embedding dim=384

Query: How do I evaluate the quality of LLM outputs?
  [0.510] LLM evaluation metrics include BLEU, ROUGE for n-gram overlap, BERTScore for sem...
  [0.314] RLHF trains a reward model on human preferences, then uses PPO to align the LLM ...
  [0.215] RAG (Retrieval-Augmented Generation) combines a retriever with an LLM generator ...


In [18]:
# ── RAG: Augment + Generate ───────────────────────────────────────────────

def rag_answer(query: str, k: int = 3, provider: str = "anthropic") -> dict:
    chunks = retrieve(query, k=k)
    context_str = "\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(chunks))

    prompt = f"""Answer the question using ONLY the provided context. 
If the context doesn't contain the answer, say "I don't have that information."

Context:
{context_str}

Question: {query}"""

    if provider == "openai":
        resp = oai.chat.completions.create(
            model=OAI_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,  # deterministic for factual retrieval
        )
        answer = resp.choices[0].message.content
    else:
        resp = ant.messages.create(
            model=ANT_MODEL,
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        answer = resp.content[0].text

    return {"answer": answer, "sources": chunks}

# Test with known and unknown questions
for q in [
    "What is RAG and how does it work?",
    "How do transformers handle long sequences?",
    "What is the capital of France?",  # not in corpus → should say no info
]:
    result = rag_answer(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"  (from {len(result['sources'])} retrieved chunks, top score: {result['sources'][0]['score']:.3f})")
    print()

Q: What is RAG and how does it work?
A: Based on the provided context, **RAG (Retrieval-Augmented Generation)** is a system that **combines a retriever with an LLM (Large Language Model) generator** to ground responses in external documents.

In other words, it works by:
1. **Retrieving** relevant information from external documents, and
2. **Generating** responses using an LLM that is grounded in that retrieved content

This approach ensures that the LLM's outputs are anchored to actual external sources rather than relying solely on its pre-trained knowledge.
  (from 3 retrieved chunks, top score: 0.426)

Q: How do transformers handle long sequences?
A: Based on the provided context, I can only partially address this question. According to the context, transformer models **use self-attention to process all tokens in parallel**, which enables parallelism during training. Additionally, the context mentions that **context window limits** exist, which drive chunking strategies (fixed-size

---
## 7. Memory & Session Management

LLMs are stateless — every call starts fresh. Strategies for maintaining conversation context:

| Strategy | Pros | Cons |
|---|---|---|
| **Full history** | Simple, lossless | Hits token limit fast |
| **Sliding window** | Bounded cost | Loses early context |
| **Summarization** | Compresses well | Summary may lose detail |
| **External memory** | Infinite scale | Retrieval adds latency |

In [19]:
# ── Pattern 1: Full history (simplest) ───────────────────────────────────
class SimpleConversation:
    def __init__(self, system: str = "You are a concise assistant."):
        self.system = system
        self.history: list[dict] = []

    def chat(self, user_msg: str) -> str:
        self.history.append({"role": "user", "content": user_msg})
        resp = ant.messages.create(
            model=ANT_MODEL,
            max_tokens=256,
            system=self.system,
            messages=self.history,
        )
        reply = resp.content[0].text
        self.history.append({"role": "assistant", "content": reply})
        return reply

    @property
    def token_estimate(self) -> int:
        return sum(len(m["content"]) // 4 for m in self.history)  # rough estimate

conv = SimpleConversation(system="You are a helpful tutor. Be concise.")
print("Q1:", conv.chat("My name is Alex. What is backpropagation?"))
print("Q2:", conv.chat("Give me a one-line analogy for it."))
print("Q3:", conv.chat("What is my name?  Did I ask about gradients?"))  # memory test
print(f"\nHistory: {len(conv.history)} messages, ~{conv.token_estimate} tokens")

Q1: Hi Alex! **Backpropagation** is the algorithm used to train neural networks.

**In short:**
1. **Forward pass** – input data flows through the network, producing a prediction
2. **Calculate error** – compare the prediction to the actual answer (loss function)
3. **Backward pass** – propagate the error *backwards* through the network, calculating how much each weight contributed to the error (using the **chain rule** of calculus)
4. **Update weights** – adjust weights to reduce the error (gradient descent)

This process repeats until the network learns accurate predictions.

**Key idea:** It efficiently computes *gradients* so we know which direction to nudge each weight to minimize error.
Q2: It's like tracing back who's to blame after a team mistake, then having everyone adjust their behavior accordingly.
Q3: Your name is **Alex**, and no — you didn't ask about gradients; I brought that up on my own in my explanation.

History: 6 messages, ~258 tokens


In [20]:
# ── Pattern 2: Sliding window (token-limited) ─────────────────────────────
import tiktoken as _tiktoken

def _count_tokens(messages: list, model: str = "gpt-4o-mini") -> int:
    enc = _tiktoken.encoding_for_model(model)
    total = 3
    for m in messages:
        total += 4
        content = m.get("content", "")
        if isinstance(content, str):
            total += len(enc.encode(content))
    return total

class SlidingWindowConversation:
    """Drops oldest messages when context exceeds max_tokens."""

    def __init__(self, max_tokens: int = 800, system: str = ""):
        self.history: list[dict] = []
        self.max_tokens = max_tokens
        self.system_messages = ([{"role": "system", "content": system}] if system else [])

    def _trim(self):
        while _count_tokens(self.system_messages + self.history) > self.max_tokens:
            if len(self.history) <= 2:
                break  # keep at least the last exchange
            self.history.pop(0)  # drop oldest message

    def chat(self, user_msg: str) -> str:
        self.history.append({"role": "user", "content": user_msg})
        self._trim()
        messages = self.system_messages + self.history
        resp = oai.chat.completions.create(
            model=OAI_MODEL, messages=messages, max_tokens=200,
        )
        reply = resp.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        return reply

sw = SlidingWindowConversation(max_tokens=400, system="You are a helpful assistant.")
topics = [
    "Explain neural networks in 2 sentences.",
    "What about transformers?",
    "And attention mechanisms?",
    "What was the first topic I asked about?"  # may be dropped from window
]
for msg in topics:
    reply = sw.chat(msg)
    print(f"Q: {msg}")
    print(f"A: {reply[:120]}")
    print(f"   [window: {len(sw.history)} msgs, ~{_count_tokens(sw.history)} tokens]\n")

Q: Explain neural networks in 2 sentences.
A: Neural networks are computational models inspired by the human brain, consisting of interconnected layers of nodes (neur
   [window: 2 msgs, ~72 tokens]

Q: What about transformers?
A: Transformers are a type of neural network architecture designed primarily for processing sequential data, such as text, 
   [window: 4 msgs, ~151 tokens]

Q: And attention mechanisms?
A: Attention mechanisms are techniques used in neural networks to allow models to focus on specific parts of the input data
   [window: 6 msgs, ~234 tokens]

Q: What was the first topic I asked about?
A: The first topic you asked about was neural networks.
   [window: 8 msgs, ~261 tokens]



In [21]:
# ── Pattern 3: Summarization-based compression ────────────────────────────
# Every N turns, compress the history into a summary and clear the raw history

class SummarizingConversation:
    def __init__(self, compress_every: int = 4):
        self.history: list[dict] = []
        self.summary: str = ""
        self.compress_every = compress_every
        self.turn_count = 0

    def _compress(self):
        if not self.history:
            return
        transcript = "\n".join(
            f"{m['role'].upper()}: {m['content']}" for m in self.history
        )
        prefix = f"Previous summary: {self.summary}\n\n" if self.summary else ""
        prompt = f"{prefix}Summarize the following conversation in 2-3 sentences, preserving all specific facts and preferences:\n\n{transcript}"
        resp = oai.chat.completions.create(
            model=OAI_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
            temperature=0,
        )
        self.summary = resp.choices[0].message.content
        self.history = []
        print(f"  [compressed → summary: '{self.summary[:80]}...']")

    def chat(self, user_msg: str) -> str:
        self.turn_count += 1
        if self.turn_count > 1 and (self.turn_count - 1) % self.compress_every == 0:
            self._compress()

        messages = []
        if self.summary:
            messages.append({"role": "system",
                             "content": f"Context from earlier in conversation: {self.summary}"})
        messages.extend(self.history)
        messages.append({"role": "user", "content": user_msg})

        resp = oai.chat.completions.create(
            model=OAI_MODEL, messages=messages, max_tokens=200,
        )
        reply = resp.choices[0].message.content
        self.history.append({"role": "user", "content": user_msg})
        self.history.append({"role": "assistant", "content": reply})
        return reply

sc = SummarizingConversation(compress_every=3)
turns = [
    "I'm Alex. I prefer Python and I'm learning about ML optimization.",
    "What's the difference between SGD and Adam?",
    "Which one should I use for fine-tuning a transformer?",
    "What language did I say I prefer?",  # should still be in summary
    "What optimizer did you recommend?",
]
for msg in turns:
    reply = sc.chat(msg)
    print(f"Q: {msg}")
    print(f"A: {reply[:150]}\n")

Q: I'm Alex. I prefer Python and I'm learning about ML optimization.
A: Great to meet you, Alex! Python is a fantastic choice for machine learning (ML), and there's a lot to explore in the area of optimization. Are you loo

Q: What's the difference between SGD and Adam?
A: Stochastic Gradient Descent (SGD) and Adam (Adaptive Moment Estimation) are both optimization algorithms commonly used for training machine learning m

Q: Which one should I use for fine-tuning a transformer?
A: When it comes to fine-tuning transformer models, Adam (especially its variant AdamW) is generally the preferred choice over Stochastic Gradient Descen

  [compressed → summary: 'Alex is learning about machine learning (ML) optimization and prefers using Pyth...']
Q: What language did I say I prefer?
A: You mentioned that you prefer using Python for learning about machine learning (ML) optimization.

Q: What optimizer did you recommend?
A: I recommended the Adam optimizer, as it is generally preferred for fine

---
## 8. Structured Output & LLM-as-Judge

**Structured output** forces the LLM to return data in a specific JSON schema — critical for pipelines that parse LLM responses programmatically.

- **OpenAI**: `response_format` with Pydantic model → guaranteed schema via `beta.chat.completions.parse`
- **Anthropic**: force a tool call with your schema as `input_schema` → tool input is always schema-valid

**LLM-as-Judge** uses a structured-output LLM call to score other LLM outputs — a core pattern in automated eval pipelines.

In [22]:
# ── OpenAI Structured Output (Pydantic) ───────────────────────────────────
from pydantic import BaseModel, Field

class SentimentResult(BaseModel):
    sentiment: str          = Field(description="'positive', 'negative', or 'neutral'")
    confidence: float       = Field(ge=0.0, le=1.0)
    key_phrases: list[str]  = Field(description="phrases driving the sentiment")
    reasoning: str          = Field(description="brief explanation")

# beta.chat.completions.parse — returns .parsed field as your Pydantic object
response = oai.beta.chat.completions.parse(
    model="gpt-4o-mini",   # structured output requires gpt-4o-mini or gpt-4o
    messages=[
        {"role": "system", "content": "Analyze the sentiment of user text."},
        {"role": "user", "content": "The new feature is fantastic but the onboarding is painful!"},
    ],
    response_format=SentimentResult,   # pass the Pydantic class directly
)

result: SentimentResult = response.choices[0].message.parsed
print(f"Sentiment:   {result.sentiment}")
print(f"Confidence:  {result.confidence:.2f}")
print(f"Key phrases: {result.key_phrases}")
print(f"Reasoning:   {result.reasoning}")
print()

# Structured output via response_format dict (JSON schema, no Pydantic)
response2 = oai.chat.completions.create(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Extract: name and age from: 'Alice is 30 years old'"}],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "entity_extraction",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "age": {"type": "integer"},
                },
                "required": ["name", "age"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
)
print("JSON schema extraction:", response2.choices[0].message.content)

Sentiment:   mixed
Confidence:  0.85
Key phrases: ['new feature is fantastic', 'onboarding is painful']
Reasoning:   The user expresses a positive sentiment towards the new feature while simultaneously stating a negative sentiment about the onboarding process, indicating mixed feelings.

JSON schema extraction: {"name":"Alice","age":30}


In [23]:
# ── Anthropic Structured Output via Tool Use ──────────────────────────────
# Force a specific tool → tool's input_schema acts as the output schema

extraction_tool = {
    "name": "submit_extraction",
    "description": "Submit the extracted structured data.",
    "input_schema": {
        "type": "object",
        "properties": {
            "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "key_phrases": {"type": "array", "items": {"type": "string"}},
            "reasoning": {"type": "string"},
        },
        "required": ["sentiment", "confidence", "key_phrases", "reasoning"],
    },
}

response = ant.messages.create(
    model=ANT_MODEL,
    max_tokens=512,
    system="Analyze the sentiment of user text and submit the result via the tool.",
    tools=[extraction_tool],
    tool_choice={"type": "tool", "name": "submit_extraction"},  # force the schema
    messages=[{"role": "user", "content": "The new feature is fantastic but onboarding is painful!"}],
)

# tool input IS the structured output
result = response.content[0].input
print(f"Sentiment:   {result['sentiment']}")
print(f"Confidence:  {result['confidence']:.2f}")
print(f"Key phrases: {result['key_phrases']}")
print(f"Reasoning:   {result['reasoning']}")

Sentiment:   neutral
Confidence:  0.85
Key phrases: ['new feature', 'fantastic', 'onboarding', 'painful']
Reasoning:   The text contains both a strong positive sentiment ("new feature is fantastic") and a strong negative sentiment ("onboarding is painful"). These opposing sentiments balance each other out, resulting in an overall neutral/mixed sentiment. The enthusiasm about the feature is countered by the frustration with the onboarding experience.


In [24]:
# ── LLM-as-Judge Pattern ──────────────────────────────────────────────────
# Used in automated eval pipelines: one LLM grades another's output

judge_tool = {
    "name": "submit_judgment",
    "description": "Submit a multi-criteria evaluation of an answer.",
    "input_schema": {
        "type": "object",
        "properties": {
            "accuracy": {
                "type": "integer", "minimum": 1, "maximum": 5,
                "description": "Factual correctness of the answer",
            },
            "completeness": {
                "type": "integer", "minimum": 1, "maximum": 5,
                "description": "Does it fully address the question?",
            },
            "clarity": {
                "type": "integer", "minimum": 1, "maximum": 5,
                "description": "Is the answer clear and well-written?",
            },
            "issues": {
                "type": "array", "items": {"type": "string"},
                "description": "List of specific problems with the answer",
            },
            "overall_feedback": {"type": "string"},
        },
        "required": ["accuracy", "completeness", "clarity", "issues", "overall_feedback"],
    },
}

JUDGE_SYSTEM = """You are an expert evaluator assessing LLM-generated answers.
Score each criterion 1-5 where 5 is perfect. Be critical and specific about issues."""

def llm_judge(question: str, answer: str) -> dict:
    response = ant.messages.create(
        model=ANT_MODEL,
        max_tokens=512,
        system=JUDGE_SYSTEM,
        tools=[judge_tool],
        tool_choice={"type": "tool", "name": "submit_judgment"},
        messages=[{
            "role": "user",
            "content": f"Question: {question}\n\nAnswer to evaluate: {answer}",
        }],
    )
    return response.content[0].input

# Evaluate a good answer
scores_good = llm_judge(
    question="What is gradient descent?",
    answer="Gradient descent is an iterative optimization algorithm that minimizes a loss function by updating parameters in the direction opposite to the gradient, scaled by the learning rate. It is the backbone of training neural networks.",
)
print("=== Good Answer ===")
print(f"Accuracy={scores_good['accuracy']}/5  Completeness={scores_good['completeness']}/5  Clarity={scores_good['clarity']}/5")
print(f"Issues: {scores_good['issues']}")
print(f"Feedback: {scores_good['overall_feedback']}")

print()

# Evaluate a poor answer
scores_bad = llm_judge(
    question="What is gradient descent?",
    answer="It's like going downhill. Models use it sometimes.",
)
print("=== Poor Answer ===")
print(f"Accuracy={scores_bad['accuracy']}/5  Completeness={scores_bad['completeness']}/5  Clarity={scores_bad['clarity']}/5")
print(f"Issues: {scores_bad['issues']}")
print(f"Feedback: {scores_bad['overall_feedback']}")

=== Good Answer ===
Accuracy=5/5  Completeness=2/5  Clarity=4/5
Issues: ['Answer is too brief and lacks depth — it only provides a surface-level definition without elaborating on key concepts.', 'Does not explain what a gradient is or why moving in the opposite direction minimizes the loss.', 'Does not mention the update rule formula (θ = θ - α∇L), which is central to understanding the algorithm.', 'No mention of the different variants of gradient descent: Batch, Stochastic (SGD), and Mini-batch gradient descent.', 'Does not explain the role or impact of the learning rate (e.g., too high vs. too low).', 'No discussion of limitations such as local minima, saddle points, or sensitivity to learning rate.', "The claim that it is 'the backbone of training neural networks' is accurate but abrupt and unexplained — it would benefit from elaboration."]
Feedback: The answer is factually correct but far too shallow for a concept as fundamental and nuanced as gradient descent. It reads more like a

In [25]:
# ── Batch evaluation with LLM judge ──────────────────────────────────────
# Run judge over multiple question/answer pairs and aggregate scores

eval_pairs = [
    {
        "question": "What is a transformer architecture?",
        "answer": "Transformers use self-attention to process all input tokens simultaneously, replacing RNNs. They consist of encoder and decoder stacks with multi-head attention and feed-forward layers.",
    },
    {
        "question": "What is overfitting?",
        "answer": "When model memorizes training data and performs poorly on new data.",
    },
    {
        "question": "Explain the bias-variance tradeoff.",
        "answer": "Bias is the error from wrong assumptions (underfitting); variance is the error from sensitivity to training data fluctuations (overfitting). Reducing one typically increases the other.",
    },
]

import pandas as pd

results = []
for pair in eval_pairs:
    scores = llm_judge(pair["question"], pair["answer"])
    results.append({
        "question": pair["question"][:50] + "...",
        "accuracy": scores["accuracy"],
        "completeness": scores["completeness"],
        "clarity": scores["clarity"],
        "avg": round((scores["accuracy"] + scores["completeness"] + scores["clarity"]) / 3, 2),
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nOverall mean score: {df['avg'].mean():.2f}/5")

                              question  accuracy  completeness  clarity  avg
What is a transformer architecture?...         4             2        3 3.00
               What is overfitting?...         5             2        3 3.33
Explain the bias-variance tradeoff....         5             2        4 3.67

Overall mean score: 3.33/5
